# HFCS: load raw Stata files
**Preservation:** `convert_missing=True` retains distinct Stata missing codes (`.`, `.a`, `.b`, etc.) rather than converting them all to `NaN`; categorical values and Stata dates are not converted automatically. The source `.dta` files and their labels are left untouched.

## 1. Imports and raw file locations

Requires `pandas` and a Jupyter Python kernel. If `pandas` is missing, run `%pip install pandas` in a separate cell.

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Iterator
import pandas as pd
import re
import json
import numbers
from pandas.io.stata import StataReader, StataMissingValue

# Save OUTSIDE the Git repository.
OUTPUT_DIR = data_root() / "HFCS_selected_raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Keys are HFCS *survey waves*, not necessarily the final folder's release year.
# For example, the 2023 wave is in the folder ending in STATA_2024.
WAVE_FOLDERS = {
    2010: ("HFCS_UDB_1_5_Stata_2010", ""),
    2014: ("HFCS_UDB_2_5_Stata_2014", "Data"),
    2017: ("HFCS_UDB_3_3_STATA_2017", "Data"),
    2021: ("HFCS_UDB_4_0_Stata_2021", ""),
    2023: ("HFCS_UDB_5_0_STATA_2024", ""),
}

# H: household; HN: household non-core; P: person; PN: person non-core;
# D: derived variables; W: replicate weight file (one per wave).
TABLES = ("h", "hn", "p", "pn", "d", "w")

# Optional: hard-code this only if automatic sibling-folder discovery fails.
# Example: HFCS_DATA_DIR = Path(r"C:\...\RA\HFCS Data Download")
HFCS_DATA_DIR = None

In [2]:
def data_root() -> Path:
    """Find the sibling 'HFCS Data Download' folder from the Jupyter cwd."""
    explicit = HFCS_DATA_DIR or os.environ.get("HFCS_DATA_DIR")
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.is_dir():
            raise FileNotFoundError(f"HFCS data folder not found: {path}")
        return path

    cwd = Path.cwd().resolve()
    # Jupyter may launch from the repo root, HFCS/, or an ancestor.
    for anchor in (cwd, *cwd.parents):
        for candidate in (anchor / "HFCS Data Download", anchor.parent / "HFCS Data Download"):
            if candidate.is_dir():
                return candidate.resolve()
    raise FileNotFoundError(
        "Cannot find 'HFCS Data Download' beside Research-Assistant-Git. "
        "Set HFCS_DATA_DIR in the first cell to its full local path."
    )


def raw_path(wave: int, table: str, imputation: int | None = None) -> Path:
    """Return the original .dta path. Never modifies the source file."""
    if wave not in WAVE_FOLDERS:
        raise ValueError(f"Unknown HFCS wave: {wave}")
    table = table.lower()
    if table not in TABLES:
        raise ValueError(f"Unknown table: {table!r}. Choose from {TABLES}")

    folder, subfolder = WAVE_FOLDERS[wave]
    folder_path = data_root() / folder
    if subfolder:
        folder_path /= subfolder

    if table == "w":
        if imputation is not None:
            raise ValueError("w.dta has no imputation suffix; use imputation=None")
        return folder_path / "w.dta"
    if imputation not in (1, 2, 3, 4, 5):
        raise ValueError("Select imputation 1 to 5 for h/hn/p/pn/d")
    return folder_path / f"{table}{imputation}.dta"

## 2. Check all expected input files

This checks **filenames only**. It does not open the Stata files or inspect any household records.

In [3]:
def file_specs(
    waves: Iterable[int] | None = None,
    tables: Iterable[str] | None = None,
    imputation: int | None = None,
) -> Iterator[tuple[int, str, int | None, Path]]:
    """Yield source file descriptions without loading them."""
    selected_waves = list(WAVE_FOLDERS) if waves is None else list(waves)
    selected_tables = list(TABLES) if tables is None else [t.lower() for t in tables]
    for wave in selected_waves:
        for table in selected_tables:
            if table == "w":
                # Replicate weights are in one wave-level file, not five files.
                if imputation is None:
                    yield wave, table, None, raw_path(wave, table)
            else:
                imps = range(1, 6) if imputation is None else [imputation]
                for imp in imps:
                    yield wave, table, imp, raw_path(wave, table, imp)


specs = list(file_specs())
inventory = pd.DataFrame([
    {
        "wave": wave,
        "table": table,
        "imputation": imp,
        "file": path.name,
        "exists": path.is_file(),
    }
    for wave, table, imp, path in specs
])

print("HFCS root:", data_root())
print(f"Expected raw .dta files: {len(specs)}")
print(f"Found: {int(inventory['exists'].sum())} | Missing: {int((~inventory['exists']).sum())}")
display(inventory.groupby(["wave", "table"], as_index=False)["exists"].agg(["sum", "count"]))
if not inventory["exists"].all():
    display(inventory.loc[~inventory["exists"]])

HFCS root: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download
Expected raw .dta files: 130
Found: 130 | Missing: 0


,wave,table,sum,count
0,2010,d,5,5
1,2010,h,5,5
2,2010,hn,5,5
3,2010,p,5,5
4,2010,pn,5,5
5,2010,w,1,1
6,2014,d,5,5
7,2014,h,5,5
8,2014,hn,5,5
9,2014,p,5,5


## 3. Read one raw table

By default, load **one imputation of one file** to check that the Stata reader works. You can change the three selection variables below. The notebook reports only dimensions and variable names, not household values.

In [4]:
def load_table(wave: int, table: str, imputation: int | None = None) -> pd.DataFrame:
    """Read all rows/columns with Stata values and extended missing codes intact."""
    path = raw_path(wave, table, imputation)
    if not path.is_file():
        raise FileNotFoundError(f"Missing HFCS file: {path}")
    return pd.read_stata(
        path,
        convert_missing=True,        # Preserve ., .a, .b, etc.
        convert_categoricals=False,  # Keep numeric codes, not value-label text
        convert_dates=False,         # Keep original Stata date representation
        preserve_dtypes=True,
    )


@dataclass
class RawTable:
    wave: int
    table: str
    imputation: int | None
    path: Path
    data: pd.DataFrame


def iter_raw_tables(
    waves: Iterable[int] | None = None,
    tables: Iterable[str] | None = None,
    imputation: int | None = None,
) -> Iterator[RawTable]:
    """Yield complete source tables one by one; no join or transformation."""
    for wave, table, imp, path in file_specs(waves, tables, imputation):
        yield RawTable(wave, table, imp, path, load_table(wave, table, imp))

In [5]:
# Example: load h1.dta from the 2023 HFCS wave.
SELECTED_WAVE = 2023
SELECTED_TABLE = "h"
SELECTED_IMPUTATION = 1  # Use None for "w"

raw_df = load_table(SELECTED_WAVE, SELECTED_TABLE, SELECTED_IMPUTATION)
print("Source:", raw_path(SELECTED_WAVE, SELECTED_TABLE, SELECTED_IMPUTATION))
print(f"Rows: {len(raw_df):,} | Columns: {len(raw_df.columns):,}")
print("Variable names:", list(raw_df.columns))
# raw_df is untouched and ready for subsequent, separate processing notebooks.
# Do not use raw_df.head() if you plan to commit this notebook with outputs.

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_6436\3999129013.py:6: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  return pd.read_stata(


Source: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download\HFCS_UDB_5_0_STATA_2024\h1.dta
Rows: 84,755 | Columns: 914
Variable names: ['id', 'survey', 'sa0010', 'sa0100', 'im0100', 'hw0010', 'ha0100', 'fha0100', 'hb0100', 'fhb0100', 'hb0100_b', 'fhb0100_b', 'hb0200', 'fhb0200', 'hb0300', 'fhb0300', 'hb0400', 'fhb0400', 'hb0410', 'fhb0410', 'hb0500', 'fhb0500', 'hb0600', 'fhb0600', 'hb0700', 'fhb0700', 'hb0800', 'fhb0800', 'hb0900', 'fhb0900', 'hb1000', 'fhb1000', 'hb1010', 'fhb1010', 'hb1101', 'fhb1101', 'hb1102', 'fhb1102', 'hb1103', 'fhb1103', 'hb1131a', 'fhb1131a', 'hb1131b', 'fhb1131b', 'hb1132a', 'fhb1132a', 'hb1132b', 'fhb1132b', 'hb1133a', 'fhb1133a', 'hb1133b', 'fhb1133b', 'hb1201a', 'fhb1201a', 'hb1201b', 'fhb1201b', 'hb1201c', 'fhb1201c', 'hb1201d', 'fhb1201d', 'hb1202a', 'fhb1202a', 'hb1202b', 'fhb1202b', 'hb1202c', 'fhb1202c', 'hb1202d', 'fhb1202d', 'hb1203a', 'fhb1203a', 'hb1203b', 'fhb1203b', 'hb1203c', 'fhb1203c'

## 4. Verify that every raw file can be read

Set `RUN_FULL_READ_AUDIT = True` to sequentially read every file and report its row and column counts. This may take a while, but it **does not** create cleaned or merged data, nor write any output files. Only one source file is held in memory at a time during this audit.

In [6]:
RUN_FULL_READ_AUDIT = True

if RUN_FULL_READ_AUDIT:
    checks = []
    for wave, table, imp, path in file_specs():
        df = load_table(wave, table, imp)
        checks.append({
            "wave": wave,
            "table": table,
            "imputation": imp,
            "file": path.name,
            "rows": len(df),
            "columns": len(df.columns),
        })
        del df
    read_audit = pd.DataFrame(checks)
    display(read_audit)
else:
    print("Full read audit skipped. Set RUN_FULL_READ_AUDIT=True to run it.")

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_6436\3999129013.py:6: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  return pd.read_stata(
C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_6436\3999129013.py:6: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  return pd.read_stata(
C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_6436\3999129013.py:6: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen w

,wave,table,imputation,file,rows,columns
0,2010,h,1.0,h1.dta,68627,778
1,2010,h,2.0,h2.dta,68627,778
2,2010,h,3.0,h3.dta,68627,778
3,2010,h,4.0,h4.dta,68627,778
4,2010,h,5.0,h5.dta,68627,778
...,...,...,...,...,...,...
125,2023,d,2.0,d2.dta,84755,218
126,2023,d,3.0,d3.dta,84755,218
127,2023,d,4.0,d4.dta,84755,218
128,2023,d,5.0,d5.dta,84755,218


## 5. Exporting the necessary variables / data for further analysis downstream

In [8]:
# Select variable families from the HFCS documentation.
# All matching variables are retained with their original names.
VARIABLE_PATTERNS = {
    "h": [
        r"sa\d{4}",       # Country, household IDs, survey metadata
        r"im0100",        # Imputation identifier
        r"hw0010",       # Household weight
        r"hh0[1-5]\d{2}", # Inheritance / gift information
        r"hi\d{4}",      # Household income information
    ],
    "d": [
        r"sa\d{4}",       # Identifiers
        r"im0100",
        r"dn\w+",        # Net wealth and related variables
        r"da\w+",        # Assets
        r"dl\w+",        # Liabilities
        r"di\w+",        # Derived income
        r"dh\w+",        # Derived household demographics
    ],
}

def select_variables(available, patterns):
    return [
        name for name in available
        if any(re.fullmatch(p, name, flags=re.I) for p in patterns)
    ]


def preserve_stata_missing(df):
    """
    Store Stata missing codes separately when exporting to Parquet.

    Example:
       Original Stata:   120000, .a, 50000
       Parquet values:   120000, null, 50000
       Missing codes:    null, ".a", null

    This permits reconstruction of the original missing codes.
    """
    df = df.copy()
    missing_columns = []

    for col in list(df.columns):
        series = df[col]

        if series.dtype != object:
            continue

        mask = series.map(
            lambda x: isinstance(x, StataMissingValue)
        )

        if not mask.any():
            continue

        code_col = f"{col}__stata_missing"

        if code_col in df.columns:
            raise ValueError(f"Column collision: {code_col}")

        codes = pd.Series(pd.NA, index=df.index, dtype="string")
        codes.loc[mask] = series.loc[mask].map(str).tolist()

        values = series.where(~mask, other=None)
        sample = next(
            (x for x in values if x is not None and not pd.isna(x)),
            None
        )

        dtype = "Int64" if isinstance(sample, numbers.Integral) else "Float64"

        df[col] = pd.array(values, dtype=dtype)
        df[code_col] = codes

        missing_columns.append(col)

    return df, missing_columns


# Process one source file at a time to limit memory usage.
audit = []

for wave in WAVE_FOLDERS:
    for imp in range(1, 6):
        for table, patterns in VARIABLE_PATTERNS.items():

            source = raw_path(wave, table, imp)

            with StataReader(
                source,
                convert_missing=True,
                convert_categoricals=False,
                convert_dates=False,
                preserve_dtypes=True,
            ) as reader:

                available = list(reader.variable_labels())
                selected = select_variables(available, patterns)

                if not selected:
                    raise ValueError(
                        f"No matching variables in {source.name}"
                    )

                df = reader.read(columns=selected)

            original_rows = len(df)

            df, missing_columns = preserve_stata_missing(df)

            output = OUTPUT_DIR / f"hfcs_{wave}_{table}_imp{imp}.parquet"

            df.to_parquet(
                output,
                engine="pyarrow",
                compression="zstd",
                index=False,
            )

            # Check that the exported file retains all rows.
            check = pd.read_parquet(output)
            assert len(check) == original_rows
            assert check.columns.equals(df.columns)

            audit.append({
                "wave": wave,
                "imputation": imp,
                "table": table,
                "rows": original_rows,
                "raw_variables": len(selected),
                "stata_missing_columns": len(missing_columns),
                "output": output.name,
            })

            print(
                f"Saved {output.name}: "
                f"{original_rows:,} rows, {len(selected)} raw variables"
            )

            del df, check

# Save an aggregate loading report.
audit_df = pd.DataFrame(audit)
audit_df.to_csv(OUTPUT_DIR / "export_audit.csv", index=False)

print(f"\nFinished: {len(audit_df)} files exported")
print("Output directory:", OUTPUT_DIR)
display(audit_df)


Saved hfcs_2010_h_imp1.parquet: 68,627 rows, 23 raw variables
Saved hfcs_2010_d_imp1.parquet: 68,627 rows, 164 raw variables
Saved hfcs_2010_h_imp2.parquet: 68,627 rows, 23 raw variables
Saved hfcs_2010_d_imp2.parquet: 68,627 rows, 164 raw variables
Saved hfcs_2010_h_imp3.parquet: 68,627 rows, 23 raw variables
Saved hfcs_2010_d_imp3.parquet: 68,627 rows, 164 raw variables
Saved hfcs_2010_h_imp4.parquet: 68,627 rows, 23 raw variables
Saved hfcs_2010_d_imp4.parquet: 68,627 rows, 164 raw variables
Saved hfcs_2010_h_imp5.parquet: 68,627 rows, 23 raw variables
Saved hfcs_2010_d_imp5.parquet: 68,627 rows, 164 raw variables
Saved hfcs_2014_h_imp1.parquet: 84,611 rows, 27 raw variables
Saved hfcs_2014_d_imp1.parquet: 84,611 rows, 170 raw variables
Saved hfcs_2014_h_imp2.parquet: 84,611 rows, 27 raw variables
Saved hfcs_2014_d_imp2.parquet: 84,611 rows, 170 raw variables
Saved hfcs_2014_h_imp3.parquet: 84,611 rows, 27 raw variables
Saved hfcs_2014_d_imp3.parquet: 84,611 rows, 170 raw variables


,wave,imputation,table,rows,raw_variables,stata_missing_columns,output
0,2010,1,h,68627,23,19,hfcs_2010_h_imp1.parquet
1,2010,1,d,68627,164,99,hfcs_2010_d_imp1.parquet
2,2010,2,h,68627,23,19,hfcs_2010_h_imp2.parquet
3,2010,2,d,68627,164,97,hfcs_2010_d_imp2.parquet
4,2010,3,h,68627,23,19,hfcs_2010_h_imp3.parquet
5,2010,3,d,68627,164,97,hfcs_2010_d_imp3.parquet
6,2010,4,h,68627,23,19,hfcs_2010_h_imp4.parquet
7,2010,4,d,68627,164,97,hfcs_2010_d_imp4.parquet
8,2010,5,h,68627,23,19,hfcs_2010_h_imp5.parquet
9,2010,5,d,68627,164,97,hfcs_2010_d_imp5.parquet


# Attaching the STATA labels to the data

In [9]:
variable_rows = []
value_rows = []
audit_rows = []

for wave in WAVE_FOLDERS:
    for table, patterns in VARIABLE_PATTERNS.items():
        for imp in range(1, 6):

            source = raw_path(wave, table, imp)

            # Read Stata metadata, NOT the full dataset
            with StataReader(source) as reader:
                variable_labels = reader.variable_labels()
                value_labels = reader.value_labels()

            selected = select_variables(
                variable_labels.keys(), patterns
            )

            for var in selected:
                variable_rows.append({
                    "wave": wave,
                    "table": table,
                    "imputation": imp,
                    "variable": var,
                    "variable_label": variable_labels.get(var, ""),
                })

                for code, label in value_labels.get(var, {}).items():
                    value_rows.append({
                        "wave": wave,
                        "table": table,
                        "imputation": imp,
                        "variable": var,
                        "code": code,
                        "value_label": label,
                    })

            audit_rows.append({
                "wave": wave,
                "table": table,
                "imputation": imp,
                "selected_variables": len(selected),
                "variables_with_labels": sum(
                    bool(variable_labels.get(v))
                    for v in selected
                ),
                "variables_with_value_labels": sum(
                    bool(value_labels.get(v))
                    for v in selected
                ),
                "labels_do_exists": (
                    source.parent / f"labels_{table.upper()}.do"
                ).exists(),
            })

# Save alongside the Parquet datasets
metadata_dir = OUTPUT_DIR / "metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(variable_rows).to_csv(
    metadata_dir / "variable_labels.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(
    value_rows,
    columns=["wave", "table", "imputation",
             "variable", "code", "value_label"]
).to_csv(
    metadata_dir / "value_labels.csv",
    index=False,
    encoding="utf-8-sig"
)

audit_df = pd.DataFrame(audit_rows)

audit_df.to_csv(
    metadata_dir / "label_audit.csv",
    index=False
)

print("Metadata saved to:", metadata_dir)
display(audit_df)


Metadata saved to: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download\HFCS_selected_raw\metadata


,wave,table,imputation,selected_variables,variables_with_labels,variables_with_value_labels,labels_do_exists
0,2010,h,1,23,23,0,True
1,2010,h,2,23,23,0,True
2,2010,h,3,23,23,0,True
3,2010,h,4,23,23,0,True
4,2010,h,5,23,23,0,True
5,2010,d,1,164,164,0,True
6,2010,d,2,164,164,0,True
7,2010,d,3,164,164,0,True
8,2010,d,4,164,164,0,True
9,2010,d,5,164,164,0,True
